# Beyond RAG — VINE (Colab)

Question in, certified decision out. The order is fixed by the paper:

    Kq = Retrieve(q, K)                    Eq 1
    Nq = Compile(q, Kq)                    Eq 1, S3.1 (model, then deterministic)
    validate(Nq)                           S3.1, before anything executes
    execute under certificate gating       Eq 4
    a = f_LM(q, C*, Pi_q)                  Eq 6, ONLY after certification

**Run order.** Section 0 always. Section 1 ONCE, after pulling new code or
changing the corpus. Sections 2 and 3 every session. Then 4 for dry runs, 5
for live runs, 6 for the measured tables, 7 for diagnostics.

**Section 8 is parked.** MUTCD-150 and the retrieval-size sweep are kept for
the record and are not part of the VINE evaluation.


## 0. Setup

Every session.

In [ ]:
# COLAB ONLY — skip this cell if running on HPRC.
import sys, os

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── WHERE EVERYTHING LIVES ─────────────────────────────────────────────────
# Data (PDF, cache, figures, page images, HF cache, snapshots, runs) lives on
# Drive. The repo is cloned to local disk and is disposable.
#
# MRAG_BASE_DIR overrides the default in config.py. It must be set BEFORE any
# mrag import, and it must be an env var so subprocesses (ingest_v4.py, run via
# !python) inherit it — sys.modules detection does not cross process boundaries.
DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"       # <- data on Drive
REPO_DIR  = "/content/Beyond_RAG_repo"                # <- code, local, disposable
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

sys.path.insert(0, REPO_DIR)
print(f"\ndata : {DRIVE_DIR}")
print(f"code : {REPO_DIR}")

# Step 0 — poppler-utils.
# Colab ships a PARTIAL poppler install: pdftotext and pdftoppm are present
# but pdftohtml is NOT. mrag/font_index.py needs pdftohtml to read the font
# a caption is set in; without it, caption anchors cannot be validated and
# body-text mentions get cropped as figures.
!apt-get install -y -qq poppler-utils
!which pdftotext pdftoppm pdftohtml pdffonts

# Step 1 — torch matched to Colab's CUDA 12.4.
# Colab ships torch 2.5.1, but transformers >=4.51 enforces CVE-2025-32434 and
# refuses to load .bin checkpoints on torch <2.6. BGE-M3 ships .bin.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchvision==0.21.0

# Step 2 — everything else
!pip install -q -r $REPO_DIR/requirements.txt

# Step 3 — pip's bulk resolver does not enforce CEILINGS when an installed
# version already satisfies the lower bound. Colab ships newer packages than
# this pipeline can use, so pin them explicitly.
!pip install -q --no-deps --force-reinstall \
    "transformers>=4.49,<4.55" \
    "huggingface_hub>=0.34,<0.35" \
    "tokenizers>=0.21,<0.22" \
    "torchao>=0.13,<0.14"

!python -c "from transformers import PreTrainedModel; print('transformers import OK')"

In [ ]:
# API keys from Colab Secrets (key icon, left sidebar).
# Keys authenticate only; select the model with CFG.set_vlm_model(...).
from google.colab import userdata
import os

def _load(env_name, *secret_names):
    for s in secret_names:
        try:
            os.environ[env_name] = userdata.get(s)
            print(f"{env_name}: loaded (secret {s!r})")
            return
        except Exception:
            continue
    print(f"{env_name}: no matching secret (skipped)")

_load("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY", "QWEN")
_load("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY")
#_load("GEMINI_API_KEY",    "GEMINI_API_KEY")

In [ ]:
import os
from pathlib import Path

old = Path("/content/drive/MyDrive/MRAG")
new = Path(os.environ["MRAG_BASE_DIR"])

def survey(p):
    if not p.exists():
        return "does not exist"
    bits = []
    for name in ("mmrag_cache_v3", "figures", "page_images", "hf_cache"):
        d = p / name
        bits.append(f"{name}={'yes' if d.exists() else 'no'}")
    pdfs = list(p.glob("*.pdf"))
    bits.append(f"pdf={pdfs[0].name if pdfs else 'MISSING'}")
    bits.append(f"snapshot={'yes' if (p/'qdrant_db.tar').exists() else 'no'}")
    return "  ".join(bits)

print(f"OLD  {old}\n     {survey(old)}\n")
print(f"NEW  {new}\n     {survey(new)}\n")

if old.exists() and not (new / "mmrag_cache_v3").exists():
    print("ACTION: rename MRAG -> Beyond_RAG in the Drive web UI, then re-run cell 0.")
    print("        Copying through Colab works but is slow — page_images alone is ~3 GB.")
elif (new / "mmrag_cache_v3").exists():
    print("Beyond_RAG already holds the cache. Nothing to do.")
else:
    print("Neither folder has a cache. Upload the MUTCD PDF to Beyond_RAG and ingest.")

### 0.1 Sanity check

In [ ]:
import os, sys
REPO_DIR = "/content/Beyond_RAG_repo"
sys.path.insert(0, REPO_DIR if os.path.isdir(REPO_DIR) else ".")

from mrag.config import CFG

print("Environment :", CFG.environment)
print("Base dir    :", CFG.base_dir, "| exists:", CFG.base_dir.exists())
print("PDF path    :", CFG.pdf_path, "| exists:", CFG.pdf_path.exists())
print("Cache dir   :", CFG.cache_dir)
print("Qdrant dir  :", CFG.qdrant_dir, "  (local SSD, snapshotted to Drive)")
print("HF cache    :", CFG.hf_home)
print("VLM provider:", CFG.vlm_provider)
print("VLM model   :", CFG.vlm_model_api if CFG.vlm_provider == "api" else CFG.vlm_model)
print("API key set :", bool(os.environ.get(CFG.api_key_env_var)))
print()
print("Image budget: max_sheets_per_figure =", CFG.max_sheets_per_figure,
      "| max_images_total =", CFG.max_images_total,
      "| max_page_images =", CFG.max_page_images)

assert str(CFG.base_dir).endswith("Beyond_RAG"), \
    f"base_dir is {CFG.base_dir} — MRAG_BASE_DIR did not take. Re-run cell 0."

try:
    import torch
    print("GPU         :", torch.cuda.get_device_name(0))
except Exception as e:
    print("GPU         : none —", e)

assert CFG.pdf_path.exists(), (
    f"No PDF at {CFG.pdf_path} or anywhere in {CFG.base_dir}. "
    f"Upload the MUTCD PDF there first."
)

## 1. Corpus — build or update

**Run this section only when the corpus changes.** Today it changes: the 22
table-note chunks minted from the verified transcription have to be appended,
and the index needs rebuilding so `authority_inferred` reaches the payloads.

Skip to section 2 on any later session.

### 1.1 Append the minted table-note chunks (run ONCE)

In [ ]:
# The 22 chunks are footnotes the extractor never read, taken from the
# verified transcription instead. Without them, six tables -- 2C-6 and
# 4C-1..4C-5 -- have footnotes that no chunk anywhere contains, so a
# certificate citing one has nothing to point at and no search can find it.
# 4C-1 note c and the 4C-4/4C-5 scope notes are in that group.
from pathlib import Path
from mrag.config import CFG
import json

NEW = CFG.cache_dir / "table_notes_new.jsonl"      # put the file here first
CH  = CFG.chunks_jsonl

assert NEW.exists(), f"{NEW} not found -- upload it to the cache directory"
have = {json.loads(l)["chunk_id"] for l in CH.open()}
add  = [json.loads(l) for l in NEW.open()]
todo = [c for c in add if c["chunk_id"] not in have]

print(f"chunks now      : {len(have)}")
print(f"in the patch    : {len(add)}")
print(f"not yet present : {len(todo)}")

if todo:
    with CH.open("a") as fh:
        for c in todo:
            fh.write(json.dumps(c) + "\n")
    print(f"appended        -> {sum(1 for _ in CH.open())} lines")
else:
    print("nothing to do; already applied")


### 1.2 Replace the table file with the linked version (run ONCE)

In [ ]:
# Every one of the 230 footnotes now carries a chunk_id, so a calculator
# certificate can cite the specific note that governs a value rather than the
# table's notes as a blob.
import sys
sys.path.insert(0, "/content/Beyond_RAG_repo")
from mrag.vine.table_data import load as load_tables
from pathlib import Path

TABLES = Path("/content/Beyond_RAG_repo/mutcd_tables.jsonl")   # or CFG.cache_dir
ts = load_tables(TABLES)
missing = [(t.table_id, f.marker) for t in ts for f in t.footnotes if not f.chunk_id]
print(f"records {len(ts)} | footnotes {sum(len(t.footnotes) for t in ts)}"
      f" | without a chunk_id {len(missing)}")
print("LINKED version" if not missing else f"OLD version -- {missing[:5]}")


### 1.3 Re-ingest

Rebuilds the index and the graph. This is the run that puts `authority_inferred` into the payloads.

In [ ]:
!cd /content/Beyond_RAG_repo && python scripts/ingest_v4.py

### 1.4 Verify the rebuild

In [ ]:
import json, collections
from mrag.config import CFG

rows = [json.loads(l) for l in CFG.chunks_jsonl.open()]
print("chunks      :", len(rows))
print("by source   :", dict(collections.Counter(r.get("source") for r in rows)))
print("minted      :", sum(1 for r in rows if r.get("origin") == "verified_transcription"))
print("authority_inferred present:",
      sum(1 for r in rows if "authority_inferred" in r), "/", len(rows))

import pickle
g = pickle.load(CFG.graph_pickle.open("rb"))
print("graph       :", g.number_of_nodes(), "nodes,", g.number_of_edges(), "edges")

# the paper's 4.1 sentence needs these six numbers
secs = sum(1 for n in g.nodes if n.startswith("section:"))
figs = sum(1 for n in g.nodes if n.startswith("figure:") and "Table" not in n)
tabs = sum(1 for n in g.nodes if n.startswith("figure:Table"))
print(f"\nfor paper 4.1: {secs} sections, {len(rows)} chunks, {figs} figures, "
      f"{tabs} tables, {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")


### 1.5 Check the payloads actually carry it

The field exists in `chunks.jsonl` already; what mattered was getting it into the vector store.

In [ ]:
from mrag.ask import init_pipeline
p = init_pipeline(load_vlm=False)
hits = p.retriever.retrieve("minimum sign size").chunks[:3]
for c in hits:
    print(f"{c['chunk_id']:<40} authority_inferred={c.get('authority_inferred')!r}")


## 2. Initialise

Every session.

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')

if CFG.vlm_provider == "api" and not os.environ.get(CFG.api_key_env_var):
    print(f"NOTE: no key in {CFG.api_key_env_var!r} for {CFG.vlm_model_api!r}. "
          f"Init will proceed; load the key or switch with CFG.set_vlm_model(...).")

from mrag.ask import init_pipeline
pipeline = init_pipeline()
print("VLM loaded :", pipeline.vlm.loaded_name if pipeline.vlm else "none")
print("KG         :", pipeline.kg.g.number_of_nodes(), "nodes,",
                      pipeline.kg.g.number_of_edges(), "edges")

### 2.1 Choose the model

In [ ]:
# ── List every VLM the config knows about, grouped by provider ──────────────
import os
from mrag.config import (
    CFG, VLM_API_MODELS, VLM_PROVIDERS, provider_of_model,
    VLM_TEXT_ONLY_ALIASES, VLM_VISION_UNVERIFIED,
)

# key present?  ->  can you actually call it
have_key = {p: bool(os.environ.get(v["env_var"])) for p, v in VLM_PROVIDERS.items()}
print("API keys loaded:", {p: ("yes" if k else "NO") for p, k in have_key.items()})
print("Currently selected:", CFG.vlm_model_api, f"[{provider_of_model(CFG.vlm_model_api)}]")
print()

# one row per distinct model id, collecting the aliases that point at it
by_model = {}
for alias, mid in VLM_API_MODELS.items():
    by_model.setdefault(mid, []).append(alias)

for prov in ("anthropic", "gemini", "dashscope"):
    rows = [(m, a) for m, a in by_model.items() if provider_of_model(m) == prov]
    if not rows:
        continue
    print(f"── {prov.upper()}   (key: {'yes' if have_key[prov] else 'NO'})")
    for mid, aliases in sorted(rows):
        flags = []
        if mid in VLM_VISION_UNVERIFIED:
            flags.append("VISION UNVERIFIED")
        if any(a in VLM_TEXT_ONLY_ALIASES for a in aliases):
            flags.append("TEXT ONLY - will 400 on images")
        if mid == CFG.vlm_model_api:
            flags.append("<< SELECTED")
        note = ("   " + " | ".join(flags)) if flags else ""
        print(f"   {'  '.join(sorted(aliases)):<42} -> {mid}{note}")
    print()

print("Select with:  CFG.set_vlm_model('fast_claude')")
print("A raw model id works too:  CFG.set_vlm_model('claude-sonnet-5')")

In [ ]:
# ── Pick a model ────────────────────────────────────────────────────────────
resolved = CFG.set_vlm_model("frontier_claude")      # cheap, for smoke tests

print("model    :", resolved)
#print("provider :", provider_of_model(resolved))
print("key set  :", bool(os.environ.get(CFG.api_key_env_var)), f"({CFG.api_key_env_var})")

# The pipeline reads CFG at call time, so no re-init needed.
#_ = ask("What shape and colour is a STOP sign?")

### 2.2 Wire the model into VINE

One callable serves the whole pipeline: the semantic parser, the LLM verifier,
the VLM verifier and the answer step. S3.3's model-agnosticism lives here and
nowhere else -- swapping provider means passing a different function.

In [ ]:
from mrag.vine import make_ask, build_verifiers
from mrag.vine.table_data import load as load_tables
from pathlib import Path

TABLES = Path("/content/Beyond_RAG_repo/mutcd_tables.jsonl")
tables = load_tables(TABLES)
ask = make_ask(pipeline.vlm, max_tokens=1200)

print("tables    :", len(tables), "records")
print("model     :", pipeline.vlm.loaded_name)
print("verifiers :", sorted(build_verifiers(tables=tables, kg=pipeline.kg,
                                            retriever=pipeline.retriever,
                                            ask=ask)))


## 3. Dry runs — compile a network, spend nothing

`dry_run=True` stops after validation. One model call for the parser, no
verifier calls, no answer call.

**Do this ten times before letting anything execute.** The semantic parser is
the component with the most freedom; every network it produces is *valid*,
which is not the same as *right*, and the difference only shows by reading
them.

In [ ]:
from mrag.vine import ask_vine

r = ask_vine(
    "Which horizontal alignment sign is required in advance of this curve?",
    retriever=pipeline.retriever,
    tables=tables, kg=pipeline.kg, ask=ask,
    dry_run=True,
)
print(r.summary())


### 3.1 Read what the parser actually produced

In [ ]:
# The obligations, in order, with the verifier each was assigned BY RULE.
for o in r.spec.obligations:
    op = r.network.op(o.id)
    guard = f"  guard: {o.guard.describe()}" if o.guard else ""
    print(f"{o.id:4} [{o.authority:9}] {op.verifier:<26} {o.type:<16}"
          f" requires={o.requires}{guard}")
    print(f"     {o.claim[:100]}")
    if o.evidence_hint:
        print(f"     hints: {o.evidence_hint}")
print()
for m in r.spec.merges:
    print(f"{m.id:4} {m.kind:<12} inputs={m.inputs}")
print(f"\nterminal: {r.spec.terminal}")


### 3.2 Was it a good decomposition?

Three things worth checking by eye, because none of them can fail validation:

1. **Are the obligations atomic?** A paragraph holding two conditions should
   be two obligations. If they are whole paragraphs, the parser is behaving
   like the baseline compiler and everything will route to the LLM.
2. **Are the dependencies real?** Two checks that could run in either order
   must not depend on each other; an invented order destroys the parallelism
   Table 3 measures.
3. **Is the exception merge the right way round?** `inputs[0]` is the base
   rule, never the exception.

In [ ]:
from collections import Counter
print("verifier split:", dict(Counter(o.verifier for o in r.network.operations
                                      if o.merge is None)))
print("parse attempts:", r.report.attempts, "| source:", r.report.source)
for group in r.report.problems:
    print("  rejected:", group)
print("\n--- raw parser reply ---")
print(r.report.raw[-1][:1500])


### 3.3 Compare against the printed structure

The baseline compiler on the same section, for reference. If the parser is not beating this, it is not earning its model call.

In [ ]:
base = ask_vine("Which horizontal alignment sign is required?",
                retriever=pipeline.retriever, tables=tables, kg=pipeline.kg,
                ask=None, dry_run=True)          # ask=None -> printed structure
print("BASELINE");  print(base.summary())
print("\nPARSER");  print(r.summary())


## 4. Live run — one question, end to end

Now verifiers run. Expect this to cost real calls: roughly one per obligation,
plus one for the parser and one for the answer.

In [ ]:
r = ask_vine(
    "Which horizontal alignment sign is required in advance of this curve?",
    retriever=pipeline.retriever,
    tables=tables, kg=pipeline.kg, ask=ask,
    record_dir=str(CFG.base_dir / "vine_runs"),
)
print(r.summary())
print("\n" + "=" * 70)
print(r.answer.text)
print("=" * 70)
print("\ncitations:")
for c in r.answer.citations:
    print("  ", c)
if r.answer.unresolved:
    print("\nleft unresolved:")
    for u in r.answer.unresolved:
        print("  ", u)


### 4.1 Read every certificate

S3.4 requires the trace to be auditable. This is that.

In [ ]:
from mrag.vine import supporting_certificates
for c in supporting_certificates(r.network, r.trace):
    print(f"[{c.status.value:7}] {c.normative_authority.value:9} "
          f"{c.verifier:<26} conf {c.confidence:.2f}")
    print(f"   {c.claim[:95]}")
    print(f"   evidence: {[(e.type, e.id) for e in c.evidence]}")
    reason = (c.provenance or {}).get("reason")
    if reason:
        print(f"   reason  : {reason}")
    print()
print("unsupported certification:", r.trace.unsupported_certification())


### 4.2 The three failures to look for

None of these makes the run crash, which is why they need looking for.

* **a verifier that answered when it should have abstained** — check any
  certificate whose evidence list is empty but whose status is not UNKNOWN.
  The model verifiers downgrade this automatically; a new verifier might not.
* **`fabricated_evidence_ids` in provenance** — the model cited something it
  was never shown.
* **`partial_figures`** — the VLM saw some sheets of a figure, not all. Its
  confidence is capped at 0.5 when this happens.

In [ ]:
for c in r.trace.store.all():
    p = c.provenance or {}
    flags = [k for k in ("fabricated_evidence_ids", "downgraded",
                         "partial_figures", "undischarged") if k in p]
    if flags:
        print(f"{c.obligation_id:6} {c.status.value:8} {flags}")
        for f in flags:
            print(f"    {f}: {p[f]}")


## 5. Batch — a set of questions

Each run writes a JSON record: the spec, every certificate, every raw model
reply, timings. Read some by hand before trusting any aggregate.

In [ ]:
QUESTIONS = [
    # put your questions here
    "Which horizontal alignment sign is required in advance of this curve?",
]

import json
from pathlib import Path
OUT = Path(CFG.base_dir) / "vine_runs"
results = []
for q in QUESTIONS:
    res = ask_vine(q, retriever=pipeline.retriever, tables=tables,
                   kg=pipeline.kg, ask=ask, record_dir=str(OUT))
    results.append(res)
    print(res.summary()); print("-" * 70)

from collections import Counter
print("\nstage     :", dict(Counter(x.stage for x in results)))
print("decision  :", dict(Counter(x.status.value for x in results)))
print("certified :", sum(x.certified for x in results), "/", len(results))
print("compiled  :", dict(Counter(x.report.source for x in results if x.report)))
print("UCR       :", sum(1 for x in results if x.trace
                         and x.trace.unsupported_certification()), "/", len(results))


### 5.1 Measuring the parser alone

With the fallback on, a parser failure is rescued by the printed structure and
becomes invisible. Turn it off when the number being reported is the parser's.

In [ ]:
strict = [ask_vine(q, retriever=pipeline.retriever, tables=tables,
                   kg=pipeline.kg, ask=ask, dry_run=True,
                   fall_back_to_baseline=False) for q in QUESTIONS]
ok = [x for x in strict if x.report and x.report.source == "semantic_parser"]
print(f"parser produced a valid network unaided: {len(ok)} / {len(strict)}")
print("attempts needed:", [x.report.attempts for x in ok])


## 6. The measured tables

Tables 3, 4 and 5 compare execution STRUCTURE, not answer quality, so they run
with simulated verifier outcomes and cost nothing. Every configuration goes
through the real `execute()`.

`terminal_accuracy` is agreement with the network's own declared logic. It is
**not** the paper's "Full credit" column, which needs real answers.

In [ ]:
import json, random
from mrag.vine.compile import compile_section, instantiate, CompileError
from mrag.vine.experiments import table3, table5, format_table
from mrag.vine.faults import table4

chunks = [json.loads(l) for l in CFG.chunks_jsonl.open()]
secs = sorted({c["section_id"] for c in chunks})
random.Random(7).shuffle(secs)

nets = []
for s in secs:
    try:
        spec = compile_section(s, chunks)
    except CompileError:
        continue
    n, p = instantiate(spec)
    if not p and len(n.operations) >= 4:
        nets.append(n)
    if len(nets) >= 200:
        break
print("networks:", len(nets))

print("\n=== TABLE 3 — linear vs dependency-aware ===")
print(format_table(table3(nets, range(15)),
                   ["terminal_accuracy", "UCR", "calls", "latency"]))
print("\n=== TABLE 5 — ablation ===")
print(format_table(table5(nets, range(15)), ["terminal_accuracy", "UCR", "PC"]))
print("\n=== TABLE 4 — fault containment ===")
print(format_table(table4(nets[:80], range(8)),
                   ["text", "numeric", "visual", "answer preserved"]))


### 6.1 Once the parser has run on real sections

`w/o guards` in Table 5 is **n/a** on baseline networks, because the printed
structure produces no guards. Re-run Table 5 over parsed networks and the row
becomes measurable. Do not report 100 there.

In [ ]:
# Collect networks from the SEMANTIC parser rather than the printed structure.
parsed = [x.network for x in strict
          if x.network and x.report and x.report.source == "semantic_parser"]
print("parsed networks:", len(parsed),
      "| carrying guards:", sum(1 for n in parsed
                                for o in n.operations if o.guard))
if parsed:
    print(format_table(table5(parsed, range(15)),
                       ["terminal_accuracy", "UCR", "PC"]))


## 7. Diagnostics

### 7.1 Per-obligation retrieval (Eq 5)

In [ ]:
r = pipeline.retriever
print("new methods:",
      hasattr(r, "retrieve_for_obligation"),
      hasattr(pipeline.store, "fetch_chunks_by_ids"),
      hasattr(pipeline.kg, "chunks_for_section"))

print("4K.04 chunks:", len(pipeline.kg.chunks_for_section("4K.04")))
print("4K.04 cites :", pipeline.kg.sections_cited_by("4K.04"))

res = r.retrieve_for_obligation(
    query="Does this push button installation comply with 4K.04?",
    obligation="the locator tone repeats at 1-second intervals",
    certificates=[{"evidence": [{"type": "section", "id": "4K.04"}]}],
)
print("\nanchors:", res.debug["anchor_sections"], "| anchor chunks:", res.debug["anchor_chunks"])
for c in res.chunks:
    print(f"  {c['section_id']:<9} {c['content_type']:<9} {c.get('text','')[:60]}")

### 7.2 Compile-time retrieval with cross-reference expansion

In [ ]:
r = pipeline.retriever
q = "minimum sizes for regulatory signs on multi-lane conventional roads"
on  = r.retrieve_for_compile(q)
off = r.retrieve_for_compile(q, expand_cross_references=False)
print("expanded kept:", on.debug["n_expanded_kept"], "of", on.debug["n_chunks"])
print("with    :", sorted({c['section_id'] for c in on.chunks}))
print("without :", sorted({c['section_id'] for c in off.chunks}))
print("gained  :", sorted({c['section_id'] for c in on.chunks} - {c['section_id'] for c in off.chunks}))

### 7.3 Table and figure notes

In [ ]:
import json, collections
rows = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
notes = [r for r in rows if r.get("source") in ("table_note", "figure_note")
         and r.get("authority_inferred")]
print(f"notes printed inside crops: {len(notes)}")
print("  from tables :", len({r['parent_id'] for r in notes if r['source']=='table_note'}), "tables")
print("  from figures:", len({r['parent_id'] for r in notes if r['source']=='figure_note'}), "figures")
print("  inferred type:", dict(collections.Counter(r["content_type"] for r in notes)))

print("\nnotes that carry an obligation (shall / should):")
for r in [x for x in notes if x["content_type"] in ("Standard", "Guidance")][:8]:
    print(f"   [{r['content_type']:<8}] {r['text'][:115]}")

### 7.4 What did retrieval return?

In [ ]:
from mrag.retrieval import Retriever

res = pipeline.retriever.retrieve("STOP sign sizes at an all-way stop")

print(f"{len(res.chunks)} chunks\n")
for c in res.chunks:
    print(f"  {c.get('section_id'):<10} {c.get('content_type'):<9} "
          f"p.{c.get('page_printed'):<5} score={c.get('score', 0):.3f}")

print(f"\n{len(res.figures)} figures")
for f in res.figures:
    n = len(f.get("image_paths") or [])
    print(f"  {f.get('figure_id'):<16} sheets={n:<3} source={f.get('source','?')}")

### 7.5 Knowledge graph shape

In [ ]:
import collections
kg = pipeline.kg
print(kg.g.number_of_nodes(), "nodes,", kg.g.number_of_edges(), "edges")
print()
print("node kinds:")
for k, v in collections.Counter(d.get("kind", "?") for _, d in kg.g.nodes(data=True)).most_common():
    print(f"  {k:<14} {v}")
print()
print("edge labels:")
for k, v in collections.Counter(d.get("label", "?") for *_, d in kg.g.edges(data=True)).most_common():
    print(f"  {k:<20} {v}")

### 7.6 Cross-references from a section

In [ ]:
# Sections cross-referenced by a given section (cites_section edges).
target = "2B.04"
refs = set()
for c in (json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()):
    if c["section_id"] == target:
        refs.update(c.get("section_refs") or [])
print(f"{target} cross-references: {sorted(refs)}")

### 7.7 Is the sparse leg alive?

In [ ]:
import json
s = json.load(open(CFG.cache_dir / "chunks_sparse.json"))
print(len(s), "entries |", sum(1 for x in s if x), "non-empty")

### 7.8 Inspect a table's crops

In [ ]:
from IPython.display import display, Image as IPImage
import json
figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
t = [f for f in figs if f["canonical_id"] == "2B-1" and f["kind"] == "Table"]
t.sort(key=lambda f: f["page_pdf"])
print(f"{len(t)} crops for Table 2B-1\n")
for f in t:
    print(f"pdf p{f['page_pdf']}  printed p{f['page_printed']}  "
          f"sheet={f.get('sheet')}/{f.get('sheet_of')}  {f['image_path'].split('/')[-1]}")
    display(IPImage(filename=f["image_path"], width=560))

### 7.9 Multi-sheet figures

The sheet label now reports the REAL total and says when only some sheets were shown.

In [ ]:
import json, collections
figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
canon = collections.defaultdict(list)
for f in figs:
    canon[(f["kind"], f["canonical_id"])].append(f)      # key on BOTH — ids collide across kinds

multi = {k: v for k, v in canon.items() if len(v) > 1}
print(f"canonical entities : {len(canon)}")
print(f"multi-sheet        : {len(multi)}")
print(f"sheets beyond first: {sum(len(v) - 1 for v in multi.values())}")
print(f"widest             : {sorted(((len(v), k[1]) for k, v in multi.items()), reverse=True)[:5]}")
print()
print(f"caps: sheets/figure={CFG.max_sheets_per_figure}  "
      f"images/request={CFG.max_images_total}  pages reserved={CFG.max_page_images}")
covered = sum(1 for v in canon.values() if len(v) <= CFG.max_sheets_per_figure)
print(f"entities fully shown under the per-figure cap: {covered}/{len(canon)}")

# ── allocation check ───────────────────────────────────────────────────────
# Selection is ROUND-ROBIN: sheet 1 of every figure, then sheet 2 of every
# figure, and so on. Greedy allocation let one 8-sheet table eat the budget and
# starve later figures, which cost more than it gained (TB009 went from full
# credit to abstaining). Every retrieved figure must keep at least sheet 1.
def simulate(sheet_counts, cap=None, per_fig=None, pages=1, reserve=None):
    cap     = cap or CFG.max_images_total
    per_fig = per_fig or CFG.max_sheets_per_figure
    reserve = CFG.max_page_images if reserve is None else reserve
    per = [list(range(min(n, per_fig))) for n in sheet_counts]
    budget = max(0, cap - min(pages, reserve))
    picked, depth = [], 0
    while len(picked) < budget and any(len(s) > depth for s in per):
        for fi, s in enumerate(per):
            if depth < len(s) and len(picked) < budget:
                picked.append((fi, depth))
        depth += 1
    shown = len({fi for fi, _ in picked})
    return len(picked), shown, len(sheet_counts)

for label, counts in [("TB009-like: 8-sheet table + 2 figures", [8, 6, 3]),
                      ("one 8-sheet table alone",               [8]),
                      ("five single-sheet figures",             [1, 1, 1, 1, 1])]:
    n, shown, total = simulate(counts)
    ok = "OK" if shown == total else "STARVED"
    print(f"  {label:<40} {n:>2} imgs, {shown}/{total} figures  {ok}")

## 8. PARKED — do not run

Kept for the record.

**MUTCD-150.** Retired as a VINE benchmark: the 150 questions are
retrieval-focused and cannot test verification structure. A new question set
is being written. The old benchmark is still useful for one thing — checking
that a few-shot prompt has not leaked a gold answer.

**The retrieval-size sweep.** `retrieve()` stays at `top_k_after_rerank = 6`.
Nothing in `mrag/vine/` calls it; VINE uses `top_k_compile_chunks = 20`,
`top_k_obligation_chunks = 12` and a 24-chunk anchor cap. Measured: no section
in the manual is larger than a network's combined retrieval budget.

## 4.3 How many chunks should retrieval return?  — PARKED, DO NOT RUN

**Kept for the record; do not use to set `top_k_after_rerank`.** Two reasons.

1. `retrieve()` with k=6 is the RAG baseline row in Table 2 of the VINE paper.
   Tuning it changes the thing the comparison is measured against.
2. It tunes on MUTCD-150, the same questions the paper reports. That is
   tuning on the test set, and there is no dev split anywhere in the repo.

The chunk-count question belongs to `retrieve_for_compile`, not to
`retrieve()`. VINE compiles obligations out of the retrieved subgraph, so a
provision absent from Kq cannot become an obligation at all — the size
question there is about closure over citations, definitions, notes and
exceptions, not about a top-k that feeds a prompt.

`top_k_after_rerank` is 6. That was chosen against the old corpus, where a
section held 4 chunks. It now holds 6, because lists are split one chunk per
item and notes are separate chunks; the gold sections behind an answerable
question hold a median of 14 chunks, up from 9.

Chunk LENGTH barely moved (median 225 -> 213 characters), so 6 chunks still
carry about the same text. What changed is how that text is divided, and
therefore how many chunks it takes to cover the evidence a question needs.

In [ ]:
# Retrieval-size sweep. NO model generation: retrieval only, so this is cheap.
# Metrics are defined here in code, because the repo has no implementation of
# the Recall@5 / evidence-sufficiency numbers quoted in the write-ups.
import json, re, time, collections
from mrag.config import CFG

GOLD = "/content/Beyond_RAG_repo/evaluation/gold/mutcd_benchmark_gold_v1_1_msdi.jsonl"
gold = [json.loads(l) for l in open(GOLD) if l.strip()]

def expand_sections(s):
    """'2B.12-2B.17' -> ['2B.12' ... '2B.17']; plain ids pass through."""
    m = re.match(r"^(\d[A-Z])\.(\d{2})-(?:\d[A-Z]\.)?(\d{2})$", s)
    if not m:
        return [s]
    a, b = int(m.group(2)), int(m.group(3))
    return [f"{m.group(1)}.{i:02d}" for i in range(a, b + 1)]

def norm_fig(x):
    return re.sub(r"\s+", " ", str(x)).strip().replace("\u2011", "-").replace("\u2013", "-")

answerable = [q for q in gold if q.get("answerable")]
print(f"{len(gold)} questions, {len(answerable)} answerable\n")

KS = [4, 6, 8, 10, 12, 16, 20]
original_k = CFG.top_k_after_rerank
rows = []
try:
    for k in KS:
        CFG.top_k_after_rerank = k
        sec_rec, full_cov, prec, chars, fig_rec = [], [], [], [], []
        t0 = time.time()
        for q in answerable:
            r = pipeline.retriever.retrieve(q["question"])
            got_secs = [c.get("section_id", "") for c in r.chunks]
            want = {s for x in (q.get("sections") or []) for s in expand_sections(x)}
            if want:
                hit = want & set(got_secs)
                sec_rec.append(len(hit) / len(want))
                full_cov.append(1.0 if hit == want else 0.0)
                prec.append(sum(1 for s in got_secs if s in want) / max(1, len(got_secs)))
            chars.append(sum(min(len(c.get("text", "")), CFG.max_chunk_chars_in_prompt)
                             for c in r.chunks))
            want_fig = {norm_fig(x) for x in (q.get("figures") or []) + (q.get("tables") or [])}
            if want_fig:
                got_fig = {norm_fig(f.get("figure_id", "")) for f in r.figures}
                fig_rec.append(len(want_fig & got_fig) / len(want_fig))
        mean = lambda v: sum(v) / len(v) if v else float("nan")
        chars.sort()
        rows.append({
            "k": k,
            "section_recall": mean(sec_rec),
            "all_gold_sections": mean(full_cov),
            "context_precision": mean(prec),
            "figure_recall": mean(fig_rec),
            "median_prompt_chars": chars[len(chars) // 2],
            "secs_per_q": (time.time() - t0) / len(answerable),
        })
        print(f"k={k:<3} section recall {rows[-1]['section_recall']:.3f} | "
              f"all gold sections {rows[-1]['all_gold_sections']:.3f} | "
              f"precision {rows[-1]['context_precision']:.3f} | "
              f"figure recall {rows[-1]['figure_recall']:.3f} | "
              f"prompt chars {rows[-1]['median_prompt_chars']:>6} | "
              f"{rows[-1]['secs_per_q']:.2f}s/q")
finally:
    CFG.top_k_after_rerank = original_k
    print(f"\ntop_k_after_rerank restored to {CFG.top_k_after_rerank}")

out = CFG.base_dir / "retrieval_size_sweep.json"
out.write_text(json.dumps(rows, indent=2))
print("saved ->", out)


## 7. Run the MUTCD-150 benchmark

In [ ]:
# The runner is a Python API, not a CLI.
import sys
REPO = "/content/Beyond_RAG_repo"
sys.path.insert(0, f"{REPO}/benchmarks/mutcd150/v1")

from mutcd_benchmark_runner import run_benchmark
from mrag.ask import ask
from mrag.config import CFG

BENCH = f"{REPO}/benchmarks/mutcd150/v1/mutcd_benchmark_questions_v1.jsonl"
OUT   = CFG.base_dir / "benchmark_runs"          # Drive/MyDrive/Beyond_RAG/benchmark_runs

paths = run_benchmark(
    CFG=CFG,
    ask_fn=ask,
    questions_path=BENCH,
    output_root=OUT,
    run_id="beyond_rag_002_roundrobin",          # NEW id — 001 used greedy allocation
    models=[{"alias": "fable", "selector": "frontier_claude", "provider": "anthropic"}],
    prompt_style="fewshot",

    # Smoke-test the regression first. Run these three, check TB009 answers
    # instead of abstaining, THEN comment this line out for the full 150.
    #question_ids=["TB008", "TB009", "TB026"],

    resume=True,
)
for k, v in paths.items():
    print(f"{k:<24} {v}")